#### YouTube Video RAG Chatbot Using Langchain

#### Necessary Imports

In [31]:
import os
from dotenv import load_dotenv
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEndpointEmbeddings,ChatHuggingFace, HuggingFaceEndpoint
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate
# Load .env file having token to access the Huggingface API
load_dotenv()

# Get Hugging Face token
token = os.getenv("HUGGINGFACEHUB_API_TOKEN")
if not token:
    raise ValueError(
        "Hugging Face API token not found. Check your .env file."
    )

#print("Token loaded:", token is not None)

#### Step 01 Indexing

In [32]:
video_id = "Gfr50f6ZBvo" # do not use full url .use just video id
try:
    # If you don’t care which language, this returns the “best” one
    api = YouTubeTranscriptApi()
    transcript_list = api.fetch(video_id, languages=["en"])

    # Flatten it to plain text
    transcript = " ".join(chunk.text for chunk in transcript_list)
    #print(transcript)

except TranscriptsDisabled:
    print("No captions available for this video.")

In [33]:
# text splitter
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = splitter.create_documents([transcript])

In [34]:
# Embeddings and vector store using Faiss
# Huggingface Embedding model
embeddings = HuggingFaceEndpointEmbeddings(
    model="sentence-transformers/all-MiniLM-L6-v2"
)

vector_store = FAISS.from_documents(
    chunks,
    embeddings
)
print('Embeddings stored in vector store FAISS')

Embeddings stored in vector store FAISS


In [35]:
#check any v
vector_store.get_by_ids(['61e45866-a18f-4e57-bfd1-1d3e20aba115'])

[]

#### Step 02 Retrievel formation

In [36]:
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 4})
retriever.invoke("what is deepmind")# example just to check retriver working or not

[Document(id='402e954f-adb0-446e-8167-bcda35869866', metadata={}, page_content="and how it works this is tough to uh ask you this question because you probably will say it's everything but let's let's try let's try to think to this because you're in a very interesting position where deepmind is the place of some of the most uh brilliant ideas in the history of ai but it's also a place of brilliant engineering so how much of solving intelligence this big goal for deepmind how much of it is science how much is engineering so how much is the algorithms how much is the data how much is the hardware compute infrastructure how much is it the software computer infrastructure yeah um what else is there how much is the human infrastructure and like just the humans interact in certain kinds of ways in all the space of all those ideas how much does maybe like philosophy how much what's the key if um uh if if you were to sort of look back like if we go forward 200 years look back what was the key 

#### Step 03 Make a prompt template

In [37]:
prompt = PromptTemplate(
    template="""
      You are a helpful assistant.
      Answer ONLY from the provided transcript context.
      If the context is insufficient, just say you don't know.

      {context}
      Question: {question}
    """,
    input_variables = ['context', 'question'])

#### Step 04 Get the question and associated retrieved documents

In [38]:
question          = "is the topic of nuclear fusion discussed in this video? if yes then what was discussed"
retrieved_docs    = retriever.invoke(question)
context_text = "\n\n".join(doc.page_content for doc in retrieved_docs) # cancatenate the page content of from all the relevent retrieved docs
#context_text

#### Step 05 Updtae the input varaibles of dynamic prompt to get final prompt

In [39]:
final_prompt = prompt.invoke({"context": context_text, "question": question})

#### Step 06 LLM Model

In [40]:
# Hugging Face model
llm = HuggingFaceEndpoint(
    repo_id="Qwen/Qwen2.5-7B-Instruct",
    task="text-generation",
    provider="featherless-ai",
    huggingfacehub_api_token=token,
    max_new_tokens=512,
    temperature=0.7
)

# Convert to LangChain chat model
model = ChatHuggingFace(llm=llm)
result=model.invoke(final_prompt)
#print(result.content)

#### Doing the same using Chain

In [41]:
# necssary imports: runable for chain and parsers for final output
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

#### runablelamda needs function to covert info form our retrived docs into single context_text

In [42]:
def format_docs(retrieved_docs):
  context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
  return context_text

In [43]:
parallel_chain = RunnableParallel({
    'context': retriever | RunnableLambda(format_docs),
    'question': RunnablePassthrough()
})

In [44]:
parser = StrOutputParser()
main_chain = parallel_chain | prompt | model | parser
main_chain.invoke('Can you summarize the video')

'The video discusses the idea of a deeper, more fundamental explanation of physics that might simplify and encompass the mysteries of consciousness, life, and gravity, rather than relying on the current standard model of physics, which is still being expanded despite known limitations. It emphasizes the importance of clear and simple explanations as a sign of true understanding. The discussion also touches on the evolution of AI and computing, referencing early chess programming efforts and the famous match between Garry Kasparov and Deep Blue, highlighting the impressive human mind as a form of intelligence that surpasses computational power in certain aspects.'